# Depth 1.1 — PC stability across seeds + PC1 confound test

**Two open questions left by Depth 1 (seed 0)**:

1. **The "important" PC moved**. The pre-existing note said PC2 carried the cross-modal
   signal (R²=0.26). Depth 1 at seed 0 found PC2 R² ≈ 0 and PC3 R² = 0.26 instead.
   PC indices are seed-dependent (and signs are arbitrary), so a single-seed table cannot
   tell us whether there is a *physical mode* that consistently has high FC-R² + MZ
   separation, or whether the "important PC" jumps around unpredictably across seeds.

2. **PC1 might be sex + brain volume in disguise**. PC1 had FC->R²=0.59, AUC_MZ=0.82,
   AUC_sibling=0.49. A demographic-shared component (sex one-hot + 16 FreeSurfer volumes)
   would look exactly like this: MZ twins are sex-matched and have very similar BV;
   DZ/siblings less so, unrelated subjects not at all. Until we residualize PC1 on
   sex+BV and check what's left, the "FC predicts a heritable structural mode" reading
   is not falsifiable.

This notebook resolves both. **Gating logic**: Section A first. If no PC has stable
high FC-R² + MZ separation across seeds, the mechanism story is dead regardless of
Section B's result.

## Plan

| Section | Question | Cost |
|---|---|---|
| A | 10-seed PC stability: align top-10 SC PCs across seeds by loading cosine similarity; for each stable mode, report median FC-R² + AUCs | ~10 min |
| B | PC1 confound test: regress PC1 scores ~ sex + brain volume at seed 0; check whether the residualized PC1 is still FC-predictable | ~10 s |
| C | Synthesis: combine the two — decide which of {real low-dim mechanism, MZ-only artifact, PC1=demographic-confound} the data supports | instant |

Output: `../model_overviews/results/local_results/further_exploration/depth1.1_pc_stability_and_confounds/`


In [1]:
# ============= SETUP =============
import sys
from pathlib import Path
for _cand in [Path.cwd(),
              Path.cwd() / "notebooks-FC_to_SC-experimental" / "further_exploration",
              Path.cwd().parent,
              Path.cwd().parent.parent]:
    if (_cand / "_setup.py").exists():
        sys.path.insert(0, str(_cand))
        break
else:
    raise RuntimeError(f"_setup.py not found from cwd={Path.cwd()}")
from _setup import *
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

K_PCA_TOP = 10              # top PCs to track
K_PCA_FC  = 256             # FC PCA dimensionality for FC->PC R² regression
N_SEEDS   = 10
SEEDS     = list(range(N_SEEDS))
out_dir   = results_dir("depth1.1_pc_stability_and_confounds")
print(f"out_dir = {out_dir}")


out_dir = /scratch/ans9868/Conn2Conn/notebooks-FC_to_SC-experimental/model_overviews/results/local_results/further_exploration/depth1.1_pc_stability_and_confounds


---

## Section A — 10-seed PC stability via loading alignment

For each seed, refit PCA(SC_train, K=10) and PCA(FC_train, K=256). For each of the
top-10 SC PCs, compute (i) FC→PC R² via BayesianRidge on the 256-D FC latent, and
(ii) AUC(MZ/DZ/sibling vs unrelated_matched) on the per-subject PC scores.

**Alignment**: PC indices are not comparable across seeds, and signs are arbitrary
(sklearn PCA flips signs unpredictably). To compare a "physical" mode across seeds,
use seed 0 as anchor and for each of its top-10 PCs, find the best match in every
other seed by **max |cosine similarity| of loading vectors**. The matched PC carries
the same physical mode (up to sign), regardless of its index.

For each anchor PC, then report:
  - median |cosine sim| across the 9 matches (stability score, in [0, 1])
  - median FC→R² across 10 seeds
  - median AUCs across 10 seeds
  - the matched PC index in each seed (so you can see how much the index moved)

**Decision rule**:
  - A PC is *stable* if median |cosine sim| ≥ 0.85 (low end of "same physical mode").
  - A *stable AND FC-predictable AND heritable* PC is the mechanism candidate.


In [2]:
# ===== Section A: per-seed PCA + alignment to seed-0 loadings =====
# Per-seed cache: loadings (top-10 SC PCs), FC->R² (per PC), AUCs (per PC), explained_var_ratio.
per_seed = {}

for seed in SEEDS:
    print(f"--- seed {seed} ---", flush=True)
    sp = load_seed_split(seed=seed)
    SC_tr_s, SC_te_s = sp["SC_train"], sp["SC_test"]
    FC_tr_s, FC_te_s = sp["FC_train"], sp["FC_test"]
    base_s, test_idx_s = sp["base"], sp["test_idx"]

    pca_sc_s = PCA(n_components=K_PCA_TOP, random_state=0).fit(SC_tr_s)
    pca_fc_s = PCA(n_components=K_PCA_FC,  random_state=0).fit(FC_tr_s)
    Z_FC_tr  = pca_fc_s.transform(FC_tr_s)
    Z_FC_te  = pca_fc_s.transform(FC_te_s)
    sc_scores_tr = pca_sc_s.transform(SC_tr_s)
    sc_scores_te = pca_sc_s.transform(SC_te_s)

    # Per-PC FC->R²
    fc_r2 = np.zeros(K_PCA_TOP, dtype=np.float64)
    for k in range(K_PCA_TOP):
        m = BayesianRidge(max_iter=300).fit(Z_FC_tr, sc_scores_tr[:, k])
        pred = m.predict(Z_FC_te)
        fc_r2[k] = r2_score(sc_scores_te[:, k], pred)

    # Per-PC AUCs
    rng = np.random.default_rng(42 + seed)  # vary pair sampling per seed
    pairs = pair_indices_by_relation(base_s.metadata_df, test_idx_s, rng, age_tol_yrs=3.0)
    auc_mz = np.full(K_PCA_TOP, np.nan)
    auc_dz = np.full(K_PCA_TOP, np.nan)
    auc_sb = np.full(K_PCA_TOP, np.nan)
    for k in range(K_PCA_TOP):
        score_vec = sc_scores_te[:, k]
        diff = np.abs(score_vec[:, None] - score_vec[None, :])
        sims_by_rel = extract_pair_sims(-diff, pairs)
        a = auc_vs_unrelated(sims_by_rel)
        auc_mz[k] = a.get("MZ", np.nan)
        auc_dz[k] = a.get("DZ", np.nan)
        auc_sb[k] = a.get("sibling", np.nan)

    per_seed[seed] = {
        "loadings":   pca_sc_s.components_.copy(),
        "expl_var":   pca_sc_s.explained_variance_ratio_.copy(),
        "fc_r2":      fc_r2,
        "auc_mz":     auc_mz,
        "auc_dz":     auc_dz,
        "auc_sb":     auc_sb,
        "pair_counts": {k: len(v) for k, v in pairs.items()},
    }
    print(f"  pair counts: {per_seed[seed]['pair_counts']}")
    print(f"  FC->R² (PC1..PC10): {[f'{x:.3f}' for x in fc_r2]}")

print("\nDone refitting 10 seeds.")


--- seed 0 ---


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  pair counts: {'MZ': 33, 'DZ': 13, 'sibling': 125, 'unrelated_matched': 171}
  FC->R² (PC1..PC10): ['0.593', '0.000', '0.256', '0.091', '0.092', '0.034', '0.051', '0.068', '0.050', '0.078']
--- seed 1 ---


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  pair counts: {'MZ': 21, 'DZ': 11, 'sibling': 118, 'unrelated_matched': 150}
  FC->R² (PC1..PC10): ['0.594', '0.046', '0.205', '0.116', '0.168', '0.123', '0.062', '0.032', '0.101', '0.040']
--- seed 2 ---


...

  pair counts: {'MZ': 21, 'DZ': 14, 'sibling': 135, 'unrelated_matched': 170}
  FC->R² (PC1..PC10): ['0.593', '-0.005', '0.268', '0.150', '0.136', '0.063', '0.073', '0.037', '0.120', '0.062']
--- seed 8 ---


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  pair counts: {'MZ': 30, 'DZ': 12, 'sibling': 120, 'unrelated_matched': 162}
  FC->R² (PC1..PC10): ['0.564', '0.041', '0.183', '0.111', '0.113', '0.087', '0.089', '0.080', '-0.001', '0.011']
--- seed 9 ---


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  pair counts: {'MZ': 22, 'DZ': 13, 'sibling': 129, 'unrelated_matched': 164}
  FC->R² (PC1..PC10): ['0.514', '-0.017', '0.297', '0.083', '0.142', '0.093', '0.132', '0.032', '-0.059', '0.001']

Done refitting 10 seeds.


In [3]:
# ===== Section A.2: align top-10 PCs to seed-0 by loading cosine similarity =====
anchor_loadings = per_seed[0]["loadings"]   # shape (K, n_edges)

# For each anchor PC k, find best-matching PC in each other seed (max |cosine sim|).
def cosine_sim_matrix(A, B):
    A = A / np.linalg.norm(A, axis=1, keepdims=True).clip(1e-12)
    B = B / np.linalg.norm(B, axis=1, keepdims=True).clip(1e-12)
    return A @ B.T

aligned_rows = []
for k in range(K_PCA_TOP):
    # Stack stats across seeds for the matched PC.
    match_indices = {0: (k, 1.0)}   # anchor seed: trivially itself, |cos|=1
    fc_r2_seeds = [per_seed[0]["fc_r2"][k]]
    mz_seeds    = [per_seed[0]["auc_mz"][k]]
    dz_seeds    = [per_seed[0]["auc_dz"][k]]
    sb_seeds    = [per_seed[0]["auc_sb"][k]]
    var_seeds   = [per_seed[0]["expl_var"][k]]
    cos_seeds   = []   # |cosine sim| of best match (excluding anchor)
    for s in SEEDS[1:]:
        sim_row = cosine_sim_matrix(anchor_loadings[k:k+1], per_seed[s]["loadings"])[0]
        j = int(np.argmax(np.abs(sim_row)))
        cos_seeds.append(float(np.abs(sim_row[j])))
        match_indices[s] = (j, float(sim_row[j]))   # signed sim too
        fc_r2_seeds.append(per_seed[s]["fc_r2"][j])
        mz_seeds.append(per_seed[s]["auc_mz"][j])
        dz_seeds.append(per_seed[s]["auc_dz"][j])
        sb_seeds.append(per_seed[s]["auc_sb"][j])
        var_seeds.append(per_seed[s]["expl_var"][j])

    aligned_rows.append({
        "anchor_pc":           k + 1,
        "median_abs_cos":      float(np.median(cos_seeds)),
        "min_abs_cos":         float(np.min(cos_seeds)),
        "median_expl_var":     float(np.median(var_seeds)),
        "median_FC_to_PC_R2":  float(np.median(fc_r2_seeds)),
        "median_AUC_MZ":       float(np.nanmedian(mz_seeds)),
        "median_AUC_DZ":       float(np.nanmedian(dz_seeds)),
        "median_AUC_sibling":  float(np.nanmedian(sb_seeds)),
        "matched_indices":     str({s: match_indices[s][0] + 1 for s in SEEDS}),  # 1-indexed
    })

aligned_df = pd.DataFrame(aligned_rows)
print("Per-anchor-PC stats (medians across 10 seeds, alignment by loading cosine):")
print(aligned_df.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

aligned_df.to_csv(out_dir / "stability_aligned_to_seed0.csv", index=False)
print(f"\nSaved -> {out_dir / 'stability_aligned_to_seed0.csv'}")

# Decision: which anchor PCs are stable?
STABLE_COS_THRESH = 0.85
stable = aligned_df[aligned_df["median_abs_cos"] >= STABLE_COS_THRESH]
print(f"\nStable anchor PCs (median |cos| >= {STABLE_COS_THRESH}):")
if len(stable) == 0:
    print("  NONE. The single-seed PC story is an artifact — the cross-modal signal is not")
    print("  spectrally concentrated.")
else:
    print(stable[["anchor_pc", "median_abs_cos", "median_FC_to_PC_R2",
                  "median_AUC_MZ", "median_AUC_DZ", "median_AUC_sibling"]]
          .to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

# Spearman across stable PCs.
if len(aligned_df) >= 3:
    for col in ("median_AUC_MZ", "median_AUC_DZ", "median_AUC_sibling"):
        v = aligned_df[col].values
        r = aligned_df["median_FC_to_PC_R2"].values
        mask = ~(np.isnan(v) | np.isnan(r))
        if mask.sum() >= 3:
            rho, p = spearmanr(v[mask], r[mask])
            print(f"  Spearman({col}, median_FC_to_PC_R2)  rho={rho:+.3f}  p={p:.3f}")


Per-anchor-PC stats (medians across 10 seeds, alignment by loading cosine):
 anchor_pc  median_abs_cos  min_abs_cos  median_expl_var  median_FC_to_PC_R2  median_AUC_MZ  median_AUC_DZ  median_AUC_sibling                                                    matched_indices
         1          0.9849       0.9812           0.0383              0.5921         0.8059         0.6194              0.4393       {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
         2          0.9345       0.8329           0.0159              0.0169         0.6400         0.4705              0.5662       {0: 2, 1: 2, 2: 2, 3: 3, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2}
         3          0.8856       0.8253           0.0150              0.2247         0.7108         0.5993              0.5810       {0: 3, 1: 3, 2: 3, 3: 2, 4: 3, 5: 3, 6: 3, 7: 3, 8: 3, 9: 3}
         4          0.8955       0.8483           0.0135              0.1204         0.6805         0.5994              0.5796       {0: 4, 1: 4, 2:

---

## Section B — PC1 confound test

PC1 at seed 0 looked like a "heritable structural mode FC can predict":
FC→R²=0.59, AUC_MZ=0.82. But MZ twins are sex-matched AND share brain volume tightly,
so a demographic component would have an MZ-only family signature like that just from
being sex+BV.

**Test**: at seed 0, regress test-set PC1 scores ~ [sex_onehot, FreeSurfer 16 volumes]
via OLS. Then:
  - If OLS R² ≈ 1.0 → PC1 IS sex+BV; the "FC predicts heritable structural mode"
    reading collapses to "FC predicts demographic confounds."
  - If OLS R² ≈ 0 → PC1 is independent of sex+BV; the structural mode is real.
  - If OLS R² intermediate → PC1 is partly demographic. Residualize and test
    whether the *residualized* PC1 is still FC-predictable.

This is the falsifier the seed-0 reading needs before anyone writes "FC predicts a
heritable structural backbone."


In [4]:
# ===== Section B: PC1 = sex + brain volume? =====
# Seed 0 (matches Depth 1).
sp0 = load_seed_split(seed=0)
SC_tr0, SC_te0 = sp0["SC_train"], sp0["SC_test"]
FC_tr0, FC_te0 = sp0["FC_train"], sp0["FC_test"]
bv_tr0, bv_te0 = sp0["bv_train"], sp0["bv_test"]
demo_tr0, demo_te0 = sp0["demo_train"], sp0["demo_test"]
base0, test_idx0 = sp0["base"], sp0["test_idx"]

pca_sc0 = PCA(n_components=10, random_state=0).fit(SC_tr0)
SC_pc_tr = pca_sc0.transform(SC_tr0)
SC_pc_te = pca_sc0.transform(SC_te0)

# Sex one-hot (already in demo_tr0 as 2 cols after age_z), but to be explicit:
sex_tr = base0.sex_oh[sp0["train_idx"]]      # one-hot, 2 cols
sex_te = base0.sex_oh[sp0["test_idx"]]
# Sex + brain volume features.
X_confound_tr = np.concatenate([sex_tr, bv_tr0], axis=1)
X_confound_te = np.concatenate([sex_te, bv_te0], axis=1)
print(f"Confound design: sex({sex_tr.shape[1]}) + bv({bv_tr0.shape[1]}) = "
      f"{X_confound_tr.shape[1]}-dim")

# Fit on TRAIN, eval R² on TEST for each of top 10 PCs.
confound_rows = []
for k in range(10):
    ols = LinearRegression().fit(X_confound_tr, SC_pc_tr[:, k])
    pred_te = ols.predict(X_confound_te)
    r2_te = r2_score(SC_pc_te[:, k], pred_te)
    pred_tr = ols.predict(X_confound_tr)
    r2_tr = r2_score(SC_pc_tr[:, k], pred_tr)
    confound_rows.append({
        "pc": k + 1,
        "confound_R2_train": r2_tr,
        "confound_R2_test":  r2_te,
    })

confound_df = pd.DataFrame(confound_rows)
print("\nOLS [sex || bv] -> SC_PC_k  (R²):")
print(confound_df.to_string(index=False, float_format=lambda x: f"{x:7.4f}"))
confound_df.to_csv(out_dir / "pc_confound_r2.csv", index=False)
print(f"\nSaved -> {out_dir / 'pc_confound_r2.csv'}")

# Residualize PC1 on [sex || bv] and re-test FC -> residualized PC1.
print("\n--- Residualize PC1 on sex+bv, retest FC predictability ---")
ols_pc1 = LinearRegression().fit(X_confound_tr, SC_pc_tr[:, 0])
pc1_resid_tr = SC_pc_tr[:, 0] - ols_pc1.predict(X_confound_tr)
pc1_resid_te = SC_pc_te[:, 0] - ols_pc1.predict(X_confound_te)

pca_fc0 = PCA(n_components=K_PCA_FC, random_state=0).fit(FC_tr0)
Z_FC_tr0 = pca_fc0.transform(FC_tr0)
Z_FC_te0 = pca_fc0.transform(FC_te0)

br_raw   = BayesianRidge(max_iter=300).fit(Z_FC_tr0, SC_pc_tr[:, 0])
r2_raw   = r2_score(SC_pc_te[:, 0], br_raw.predict(Z_FC_te0))
br_resid = BayesianRidge(max_iter=300).fit(Z_FC_tr0, pc1_resid_tr)
r2_resid = r2_score(pc1_resid_te,    br_resid.predict(Z_FC_te0))

print(f"  FC -> PC1 (raw)         R² = {r2_raw:.4f}")
print(f"  FC -> PC1 (resid s+bv)  R² = {r2_resid:.4f}")
print(f"  drop = {r2_raw - r2_resid:+.4f}")

# Also recompute family AUCs on residualized PC1 to see if the MZ signal survives.
rng = np.random.default_rng(42)
pairs0 = pair_indices_by_relation(base0.metadata_df, test_idx0, rng, age_tol_yrs=3.0)

def per_pc_aucs(vec, pairs):
    diff = np.abs(vec[:, None] - vec[None, :])
    sims_by_rel = extract_pair_sims(-diff, pairs)
    return auc_vs_unrelated(sims_by_rel)

a_raw   = per_pc_aucs(SC_pc_te[:, 0], pairs0)
a_resid = per_pc_aucs(pc1_resid_te,   pairs0)
print(f"\n  PC1 AUCs (raw):       {a_raw}")
print(f"  PC1 AUCs (residualized): {a_resid}")


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Confound design: sex(2) + bv(16) = 18-dim

OLS [sex || bv] -> SC_PC_k  (R²):
 pc  confound_R2_train  confound_R2_test
  1             0.8876            0.8914
  2             0.0630            0.0790
  3             0.0591           -0.0619
  4             0.1157           -0.0233
  5             0.0888            0.0351
  6             0.3352            0.2618
  7             0.2295            0.0597
  8             0.0525            0.0310
  9             0.0965            0.0886
 10             0.0575            0.0472

Saved -> /scratch/ans9868/Conn2Conn/notebooks-FC_to_SC-experimental/model_overviews/results/local_results/further_exploration/depth1.1_pc_stability_and_confounds/pc_confound_r2.csv

--- Residualize PC1 on sex+bv, retest FC predictability ---
  FC -> PC1 (raw)         R² = 0.5928
  FC -> PC1 (resid s+bv)  R² = 0.0171
  drop = +0.5757

  PC1 AUCs (raw):       {'MZ': 0.8222576643629276, 'DZ': 0.5038236617183987, 'sibling': 0.48832748538011694}
  PC1 AUCs (residualized):

---

## Section C — Synthesis: what does the data actually support?

Combines Section A (PC stability) and Section B (PC1 confound) into a single verdict.

Decision tree (executed below):

```
IF Section A finds no stable PC with high FC-R² + MZ separation:
  -> "Cross-modal signal is distributed, not spectrally concentrated." Mechanism = REJECTED.

ELIF Section B finds PC1 confound R² > 0.7 AND residualized PC1 FC-R² < 0.10:
  -> "PC1 was sex+BV in disguise; FC predicts a demographic confound."
     The 'heritable structural mode' reading was an artifact.

ELIF stable PCs exist AND PC1 confound R² < 0.4:
  -> "FC predicts a stable structural mode that is heritable beyond demographics."
     Mechanism = CONFIRMED.

ELSE:
  -> "Mechanism is partial." Report the stable+heritable PCs but flag the demographic
     overlap. Cross-modal signal exists but is entangled with anatomical priors.
```


In [5]:
# ===== Section C: synthesis verdict =====
# Pull the numbers from Sections A and B.
stable_mech_PCs = aligned_df[
    (aligned_df["median_abs_cos"] >= 0.85) &
    (aligned_df["median_FC_to_PC_R2"] >= 0.15) &
    (aligned_df["median_AUC_MZ"] >= 0.60)
]
print("Stable + FC-predictable + MZ-separating PCs:")
if len(stable_mech_PCs) == 0:
    print("  NONE.")
else:
    print(stable_mech_PCs[["anchor_pc", "median_abs_cos", "median_FC_to_PC_R2",
                            "median_AUC_MZ", "median_AUC_DZ", "median_AUC_sibling"]]
          .to_string(index=False, float_format=lambda x: f"{x:7.4f}"))

pc1_conf_r2  = confound_df.query("pc == 1").iloc[0]["confound_R2_test"]
pc1_fc_drop  = r2_raw - r2_resid

print(f"\nPC1 confound (test) R² (sex+bv -> PC1)   = {pc1_conf_r2:.4f}")
print(f"PC1 FC predictability  raw vs residualized = {r2_raw:.4f} vs {r2_resid:.4f}  (drop {pc1_fc_drop:+.4f})")

print("\n" + "=" * 70)
print("AUTOMATED VERDICT")
print("=" * 70)

if len(stable_mech_PCs) == 0:
    print("  -> NO stable PC has high FC-R² + MZ AUC across 10 seeds.")
    print("     The cross-modal signal is DISTRIBUTED, not spectrally concentrated.")
    print("     Mechanism story REJECTED. The asymmetry is real but does not factor")
    print("     into a low-dim heritable backbone.")
elif pc1_conf_r2 > 0.70 and r2_resid < 0.10 and 1 in stable_mech_PCs["anchor_pc"].values:
    print("  -> PC1 IS largely sex+BV (confound R² > 0.70). After residualization,")
    print(f"     FC predictability collapses ({r2_raw:.3f} -> {r2_resid:.3f}).")
    print("     If PC1 was the only 'mechanism' candidate, the heritable-mode reading")
    print("     was a demographic confound. Check the other stable PCs separately.")
elif len(stable_mech_PCs) > 0 and pc1_conf_r2 < 0.40:
    print("  -> Stable PCs with high FC-R² + MZ separation exist, and PC1 is NOT")
    print("     a strong sex+BV confound. Mechanism CONFIRMED: FC predicts a")
    print("     stable structural mode heritable beyond demographics.")
else:
    print("  -> Mechanism PARTIAL. Stable+heritable+FC-predictable PCs exist but")
    print("     overlap with demographic priors. Cross-modal signal is real but")
    print("     entangled with anatomy. Report both numbers, do not over-claim.")

print("\nDetail: matched-index drift across seeds (column = seed, value = matched PC):")
print(aligned_df[["anchor_pc", "matched_indices"]].to_string(index=False))


Stable + FC-predictable + MZ-separating PCs:
 anchor_pc  median_abs_cos  median_FC_to_PC_R2  median_AUC_MZ  median_AUC_DZ  median_AUC_sibling
         1          0.9849              0.5921         0.8059         0.6194              0.4393
         3          0.8856              0.2247         0.7108         0.5993              0.5810

PC1 confound (test) R² (sex+bv -> PC1)   = 0.8914
PC1 FC predictability  raw vs residualized = 0.5928 vs 0.0171  (drop +0.5757)

AUTOMATED VERDICT
  -> PC1 IS largely sex+BV (confound R² > 0.70). After residualization,
     FC predictability collapses (0.593 -> 0.017).
     If PC1 was the only 'mechanism' candidate, the heritable-mode reading
     was a demographic confound. Check the other stable PCs separately.

Detail: matched-index drift across seeds (column = seed, value = matched PC):
 anchor_pc                                                    matched_indices
         1       {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
         2 

---

## Section D — PC3 network localization (10 seeds)

PC3 is currently an abstract: a 64,620-dim loading vector that's stable, heritable,
FC-predictable, and not a sex+BV confound. This section answers *what it physically is*.

**The lookup tables we need**:
- Edge → (region_i, region_j): the upper-triangle indexing of the 360×360 region matrix
  (already implicit in how SC was vectorized; 360·359/2 = 64,620 ✓).
- Region → Yeo7 network + hemisphere: `data/atlas_info/Glasser_dseg_reformatted.csv`
  (Glasser 2016 + Cole-Anticevic remap; 7 networks: default mode, dorsal attention,
  frontoparietal, limbic, somatosensory, ventral attention, visual).

**Per seed**, we align PC3 to seed-0 by max |cosine sim| of loadings (carries the
physical mode regardless of index/sign), then compute:

| Probe | Question |
|---|---|
| **Network enrichment** | For each (netA, netB) pair, observed/expected ratio in top-K edges. > 2× = enriched. |
| **Interhemispheric fraction** | Fraction of top-K edges crossing L↔R. Elevated → callosal mode. |
| **Rich-club fraction** | Fraction of top-K edges between top-20% SC-strength hubs. Elevated → rich-club mode. |
| **Energy concentration** | Fraction of total L2 energy in top-1% of edges. > 0.30 = peaked; ~ 0.01 = uniform smear. |

**Verdict thresholds** (all three must hold for outcome (a) HEADLINE):
- A handful of network pairs at enrichment > 2× AND consistent across seeds, AND
- Either interhemispheric or rich-club fraction clearly elevated vs baseline, AND
- Top-1% energy fraction > 0.10 (i.e. not a uniform smear).

Outcomes:
- **(a) HEADLINE** — "PC3 is the [callosal/rich-club/named] structural backbone"
- **(b) DISTRIBUTED** — enrichments elevated but spread; no single clean signature
- **(c) SMEAR** — flat enrichment, uniform energy → no anatomical focality

We compute K ∈ {100, 200, 500} for sensitivity (report K=200 as primary).


In [ ]:
# ===== Section D.1: per-seed PC3 alignment + localization probes =====
ATLAS_PATH = next(p for p in [
    Path("/scratch/ans9868/Conn2Conn/data/atlas_info/Glasser_dseg_reformatted.csv"),
    Path.cwd() / "data" / "atlas_info" / "Glasser_dseg_reformatted.csv",
    Path("/Users/user/projects/Conn2Conn/data/atlas_info/Glasser_dseg_reformatted.csv"),
] if p.exists())
atlas = pd.read_csv(ATLAS_PATH)
assert len(atlas) == 360, f"Expected 360 Glasser regions, got {len(atlas)}"
# Atlas id is 1-indexed; map to 0-indexed region_i / region_j.
atlas["idx0"] = atlas["id"].astype(int) - 1
atlas_idx = atlas.set_index("idx0")
NETWORK_COL = "community_yeo"
print(f"Atlas: {len(atlas)} regions, networks = {sorted(atlas[NETWORK_COL].unique())}")
print(f"Hemispheres: {atlas['hemisphere'].value_counts().to_dict()}")

# Edge index -> (region_i, region_j).
N_REGIONS = 360
triu_i, triu_j = np.triu_indices(N_REGIONS, k=1)
N_EDGES = len(triu_i)
assert N_EDGES == 64620

# Precompute base rates on the FULL connectome edge set (network pairs, interhemi).
hemi_i = atlas_idx.loc[triu_i, "hemisphere"].values
hemi_j = atlas_idx.loc[triu_j, "hemisphere"].values
net_i  = atlas_idx.loc[triu_i, NETWORK_COL].values
net_j  = atlas_idx.loc[triu_j, NETWORK_COL].values
# net_pair: tuple(sorted([netA, netB])) string for grouping
net_pair_all = np.array([" || ".join(sorted([a, b])) for a, b in zip(net_i, net_j)])
all_pair_counts = pd.Series(net_pair_all).value_counts()
interhemi_base = float((hemi_i != hemi_j).mean())
print(f"\nBase rates over all {N_EDGES} edges:")
print(f"  interhemispheric fraction = {interhemi_base:.4f}")
print(f"  top 5 network pairs (by raw count):")
for k, v in all_pair_counts.head(5).items():
    print(f"    {k:60s}  n={v}  ({v/N_EDGES:.3f})")

# Seed 0 anchor PC3.
anchor_loadings = per_seed[0]["loadings"]
anchor_pc3 = anchor_loadings[2]                 # 0-indexed, third PC
# Orient: positive lead-edge for readability.
if anchor_pc3[np.argmax(np.abs(anchor_pc3))] < 0:
    anchor_pc3 = -anchor_pc3

# Per-seed PC3 alignment + probes.
K_LIST = [100, 200, 500]
loc_rows = []     # one row per (seed, K)
enr_rows = []     # one row per (seed, K, net_pair) for stable-pair detection
pc3_aligned = {}  # seed -> aligned PC3 loading vector

# Need each seed's SC_train mean strength per node for rich-club hubs.
# Use cached per_seed if loadings stored; recompute node strength per seed.
print("\n--- per-seed PC3 alignment + localization ---")
for s in SEEDS:
    L = per_seed[s]["loadings"]
    # find match to anchor_pc3 (across all top-10 PCs of seed s, signed)
    sims = (L @ anchor_pc3) / (np.linalg.norm(L, axis=1) * np.linalg.norm(anchor_pc3) + 1e-12)
    j = int(np.argmax(np.abs(sims)))
    sign = float(np.sign(sims[j]))
    pc3_s = sign * L[j]   # oriented to match anchor

    # Re-orient for readability (positive lead edge).
    if pc3_s[np.argmax(np.abs(pc3_s))] < 0:
        pc3_s = -pc3_s
    pc3_aligned[s] = pc3_s

    # Recompute SC node strength on this seed's TRAIN set for rich-club.
    sp_s = load_seed_split(seed=s)
    SC_tr_s = sp_s["SC_train"]
    # Reconstruct full 360x360 from upper triangle, take row-mean as node strength.
    sc_mean_edge = SC_tr_s.mean(axis=0)   # (64620,)
    sym = np.zeros((N_REGIONS, N_REGIONS), dtype=np.float32)
    sym[triu_i, triu_j] = sc_mean_edge
    sym[triu_j, triu_i] = sc_mean_edge
    node_strength = sym.mean(axis=1)
    hub_thresh = np.quantile(node_strength, 0.80)   # top 20%
    is_hub = node_strength >= hub_thresh
    richclub_base = float((is_hub[triu_i] & is_hub[triu_j]).mean())

    # Energy concentration index (seed-level, K-independent).
    sq = pc3_s ** 2
    sq_sorted = np.sort(sq)[::-1]
    n_top1pct = max(1, N_EDGES // 100)
    energy_top1pct = float(sq_sorted[:n_top1pct].sum() / sq_sorted.sum())

    for K in K_LIST:
        order = np.argsort(-np.abs(pc3_s))
        top_idx = order[:K]
        ti, tj = triu_i[top_idx], triu_j[top_idx]
        # Probes
        interhemi_frac = float((atlas_idx.loc[ti, "hemisphere"].values
                                 != atlas_idx.loc[tj, "hemisphere"].values).mean())
        richclub_frac = float((is_hub[ti] & is_hub[tj]).mean())
        # Network pair enrichment
        top_pair = np.array([" || ".join(sorted([a, b]))
                              for a, b in zip(atlas_idx.loc[ti, NETWORK_COL].values,
                                              atlas_idx.loc[tj, NETWORK_COL].values)])
        top_pair_counts = pd.Series(top_pair).value_counts()
        for pair, n_obs in top_pair_counts.items():
            n_exp = all_pair_counts.get(pair, 0) * (K / N_EDGES)
            enr = (n_obs / n_exp) if n_exp > 0 else np.nan
            enr_rows.append({"seed": s, "K": K, "net_pair": pair,
                              "n_obs": int(n_obs), "n_exp": float(n_exp),
                              "enrichment": float(enr)})
        loc_rows.append({
            "seed": s, "K": K,
            "anchor_match_signedcos": float(sims[j]),
            "anchor_match_pc_idx_1based": j + 1,
            "interhemi_frac_top":   interhemi_frac,
            "interhemi_frac_base":  interhemi_base,
            "richclub_frac_top":    richclub_frac,
            "richclub_frac_base":   richclub_base,
            "energy_top1pct_frac":  energy_top1pct,
        })
    print(f"  seed {s:2d}: matched anchor PC3 at idx {j+1} (signed cos = {sims[j]:+.3f}), "
          f"energy_top1%={energy_top1pct:.3f}")

loc_df = pd.DataFrame(loc_rows)
enr_df = pd.DataFrame(enr_rows)
loc_df.to_csv(out_dir / "pc3_localization_per_seed.csv", index=False)
enr_df.to_csv(out_dir / "pc3_enrichment_per_seed.csv", index=False)
print(f"\nSaved -> {out_dir / 'pc3_localization_per_seed.csv'}")
print(f"Saved -> {out_dir / 'pc3_enrichment_per_seed.csv'}")


In [ ]:
# ===== Section D.2: cross-seed aggregation + verdict =====
PRIMARY_K = 200

# (1) Localization probe medians across seeds.
prim = loc_df[loc_df["K"] == PRIMARY_K]
print(f"=== Localization probes (K={PRIMARY_K}, n={len(prim)} seeds) ===")
print(f"  interhemi   top: median={prim['interhemi_frac_top'].median():.4f}  "
      f"range=[{prim['interhemi_frac_top'].min():.4f}, {prim['interhemi_frac_top'].max():.4f}]")
print(f"  interhemi   base = {prim['interhemi_frac_base'].iloc[0]:.4f}")
print(f"  richclub    top: median={prim['richclub_frac_top'].median():.4f}  "
      f"range=[{prim['richclub_frac_top'].min():.4f}, {prim['richclub_frac_top'].max():.4f}]")
print(f"  richclub    base: median={prim['richclub_frac_base'].median():.4f}")
print(f"  energy top-1%: median={prim['energy_top1pct_frac'].median():.4f}  "
      f"range=[{prim['energy_top1pct_frac'].min():.4f}, {prim['energy_top1pct_frac'].max():.4f}]")

# (2) Network-pair enrichment: which pairs are STABLE across seeds?
#     Require min_enrichment across seeds > 1.5 AND median > 2.0 AND seen in >= 8 seeds.
prim_enr = enr_df[enr_df["K"] == PRIMARY_K]
agg = prim_enr.groupby("net_pair").agg(
    n_seeds=("seed", "nunique"),
    median_enrichment=("enrichment", "median"),
    min_enrichment=("enrichment", "min"),
    median_n_obs=("n_obs", "median"),
).reset_index().sort_values("median_enrichment", ascending=False)
agg.to_csv(out_dir / "pc3_enrichment_agg.csv", index=False)
print(f"\n=== Top network-pair enrichment (median across {len(SEEDS)} seeds, K={PRIMARY_K}) ===")
print(agg.head(10).to_string(index=False, float_format=lambda x: f"{x:7.3f}"))

stable_enriched = agg[(agg["n_seeds"] >= 8) &
                      (agg["min_enrichment"] >= 1.5) &
                      (agg["median_enrichment"] >= 2.0)]
print(f"\n=== STABLE enriched pairs (>=8 seeds, min>=1.5x, median>=2x) ===")
if len(stable_enriched) == 0:
    print("  NONE.")
else:
    print(stable_enriched.to_string(index=False, float_format=lambda x: f"{x:7.3f}"))

# (3) Top edges from seed 0 PC3 — region-level peek.
pc3_0 = pc3_aligned[0]
top200_idx = np.argsort(-np.abs(pc3_0))[:50]
top_edges_df = pd.DataFrame({
    "rank": np.arange(1, 51),
    "region_i_idx": triu_i[top200_idx],
    "region_j_idx": triu_j[top200_idx],
    "region_i": atlas_idx.loc[triu_i[top200_idx], "label"].values,
    "region_j": atlas_idx.loc[triu_j[top200_idx], "label"].values,
    "net_i":    atlas_idx.loc[triu_i[top200_idx], NETWORK_COL].values,
    "net_j":    atlas_idx.loc[triu_j[top200_idx], NETWORK_COL].values,
    "hemi_i":   atlas_idx.loc[triu_i[top200_idx], "hemisphere"].values,
    "hemi_j":   atlas_idx.loc[triu_j[top200_idx], "hemisphere"].values,
    "loading":  pc3_0[top200_idx],
})
top_edges_df.to_csv(out_dir / "pc3_top50_edges_seed0_labeled.csv", index=False)
print(f"\n=== Top 15 PC3 edges (seed 0, region labels) ===")
print(top_edges_df.head(15)[["region_i", "region_j", "net_i", "net_j",
                              "hemi_i", "hemi_j", "loading"]]
      .to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

# (4) VERDICT.
print("\n" + "=" * 78)
print("PC3 LOCALIZATION VERDICT")
print("=" * 78)
interhemi_elev = prim["interhemi_frac_top"].median() > 1.30 * prim["interhemi_frac_base"].iloc[0]
richclub_elev  = prim["richclub_frac_top"].median()  > 1.30 * prim["richclub_frac_base"].median()
energy_peaked  = prim["energy_top1pct_frac"].median() > 0.10
n_stable_pairs = len(stable_enriched)

print(f"  interhemi elevated (>1.3x base): {interhemi_elev}  "
      f"({prim['interhemi_frac_top'].median():.3f} vs base {prim['interhemi_frac_base'].iloc[0]:.3f})")
print(f"  richclub  elevated (>1.3x base): {richclub_elev}  "
      f"({prim['richclub_frac_top'].median():.3f} vs base {prim['richclub_frac_base'].median():.3f})")
print(f"  energy top-1% > 0.10           : {energy_peaked}  "
      f"({prim['energy_top1pct_frac'].median():.3f})")
print(f"  stable enriched pairs          : {n_stable_pairs}")
print()

if not energy_peaked and n_stable_pairs == 0:
    print("  -> OUTCOME (c) SMEAR: PC3 has no coherent network localization.")
    print("     Energy is near-uniform across edges and no network pair is stably enriched.")
    print("     The heritable cross-modal signal is statistical, not anatomically focal.")
elif n_stable_pairs >= 1 and (interhemi_elev or richclub_elev):
    sig = "interhemispheric/callosal" if interhemi_elev else "rich-club/hub"
    print(f"  -> OUTCOME (a) HEADLINE: PC3 is a {sig} structural mode.")
    print(f"     {n_stable_pairs} network pair(s) consistently enriched across seeds,")
    print(f"     AND {sig} fraction clearly elevated vs baseline.")
    if len(stable_enriched) > 0:
        print(f"     Stable enriched pairs:")
        for _, r in stable_enriched.iterrows():
            print(f"       - {r['net_pair']}: median enrichment {r['median_enrichment']:.2f}x")
elif n_stable_pairs >= 1:
    print("  -> OUTCOME (b) DISTRIBUTED: PC3's heavy edges are enriched in specific")
    print("     network pairs but neither interhemispheric nor rich-club structure")
    print("     is clearly elevated. Report as 'distributed across [networks]' rather")
    print("     than a single anatomical backbone.")
else:
    print("  -> OUTCOME mixed: no clean signature. Read the enrichment table above")
    print("     and the top-edge list manually. The data does not fit any clean category.")
